# Baselines sur le nouveau dataset : Food-101 (10 classes)

Ce notebook reprend exactement le protocole du notebook `03_run_baselines.ipynb`
(HF / BF / MIXTE) mais sur le dataset **Food-101** préparé par
le notebook `14_Prepare_Food101_Dataset.ipynb`.

L'objectif est de vérifier que les conclusions tirées sur Animals-10 et Imagewoof
se généralisent à un domaine **texturé très différent** (classification fine-grained
de plats : pizza, hamburger, hot_dog, sushi, ice_cream, french_fries, ramen, steak,
waffles, fried_rice).

## Extraire le fichier Zip (Food101) sur le SSD rapide

In [ ]:
import sys, os, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

import torch
import train_baselines
importlib.reload(train_baselines)
from train_baselines import run_baseline
print('GPU:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Test des baselines avec 1 epoch (sanity-check)

In [ ]:
# Sanity-check Food101 : 1 epoque, 1 seed, sans W&B
for m in ['HF', 'BF', 'MIXTE']:
    run_baseline(mode=m, epochs=1, dataset_name='Food101', use_wandb=False, seeds=(42,))
print('Sanity-check Food101 OK.')

## Entraînement complet des baselines (20 epochs)

In [ ]:
# Entrainement complet Food101 : 20 epoques, 3 seeds
NB_EPOCHS = 20
for m in ['HF', 'BF', 'MIXTE']:
    run_baseline(mode=m, epochs=NB_EPOCHS, dataset_name='Food101', use_wandb=True)
print('Baselines Food101 terminees.')

## Archivage des résultats Food101 dans un sous-dossier dédié

`train_baselines.py` écrit directement dans `results/Food101/` via `env_config.results_dir('Food101')`.
Aucun déplacement de fichier n'est nécessaire.

In [ ]:
import sys, os, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

print('Resultats Food101 dans:', env_config.results_dir('Food101'))

## Plot des résultats

Relecture directe des fichiers JSON produits.

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np

import sys
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
food101_dir = env_config.results_dir('Food101')

baseline_keys = ['HF', 'BF', 'MIXTE']
baseline_labels = {'HF': 'BL 1(HF)', 'BF': 'BL 2(BF)', 'MIXTE': 'BL 3(MIXTE)'}

data = {}
for k in baseline_keys:
    path = os.path.join(food101_dir, f'results_baseline_{k}.json')
    with open(path, 'r') as f:
        data[k] = json.load(f)

losses = {k: data[k].get('loss_history', []) for k in baseline_keys}
accuracies = {
    k: [data[k]['accuracy_HF'], data[k]['accuracy_BF'], data[k]['accuracy_Mixte']]
    for k in baseline_keys
}
train_time_min = {k: data[k]['training_time_sec'] / 60.0 for k in baseline_keys}
cost_total_ca  = {k: data[k]['total_cost_CA'] for k in baseline_keys}

acc_labels = ['Test HF', 'Test BF', 'Test Mixte']
has_losses = any(len(losses[k]) > 0 for k in baseline_keys)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Comparaison des 3 baselines — Food-101 (10 classes)', fontsize=16)

ax = axes[0, 0]
if has_losses:
    for k in baseline_keys:
        y = losses[k]
        if len(y) > 0:
            ax.plot(np.arange(1, len(y) + 1), y, marker='o', linewidth=2, label=baseline_labels[k])
    ax.legend()
else:
    ax.text(0.5, 0.5, 'loss_history absent du JSON', ha='center', va='center',
            transform=ax.transAxes, fontsize=11, color='gray')
ax.set_title('Loss par époque')
ax.set_xlabel('Époque')
ax.set_ylabel('Loss')
ax.grid(alpha=0.3)

ax = axes[0, 1]
x = np.arange(len(acc_labels))
width = 0.25
for i, k in enumerate(baseline_keys):
    vals = accuracies[k]
    bars = ax.bar(x + (i - 1) * width, vals, width=width, label=baseline_labels[k])
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.6, f'{h:.1f}%',
                ha='center', va='bottom', fontsize=8)
ax.set_title('Précisions sur les jeux de test')
ax.set_xticks(x)
ax.set_xticklabels(acc_labels)
ax.set_ylabel('Précision (%)')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
ax.legend()

ax = axes[1, 0]
time_vals = [train_time_min[k] for k in baseline_keys]
bars = ax.bar([baseline_labels[k] for k in baseline_keys], time_vals,
              color=['#4C72B0', '#55A868', '#C44E52'])
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h, f'{h:.2f} min',
            ha='center', va='bottom', fontsize=9)
ax.set_title("Temps d'entraînement")
ax.set_ylabel('Minutes')
ax.set_ylim(0, max(time_vals) * 1.15 if max(time_vals) > 0 else 1)
ax.grid(axis='y', alpha=0.3)

ax = axes[1, 1]
cost_vals = [cost_total_ca[k] for k in baseline_keys]
bars = ax.bar([baseline_labels[k] for k in baseline_keys], cost_vals,
              color=['#8172B2', '#CCB974', '#64B5CD'])
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h,
            f'{int(h):,} CA'.replace(',', ' '), ha='center', va='bottom', fontsize=9)
ax.set_title('Coût total estimé')
ax.set_ylabel('CA')
ax.set_ylim(0, max(cost_vals) * 1.15 if max(cost_vals) > 0 else 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
os.makedirs(food101_dir, exist_ok=True)
plt.savefig(os.path.join(food101_dir, 'baselines_comparison_food101.png'), dpi=180, bbox_inches='tight')
plt.show()

print('\n=== RECAPITULATIF FOOD101 ===')
for k in baseline_keys:
    print(f'{baseline_labels[k]:<12} | HF={accuracies[k][0]:5.2f}%  BF={accuracies[k][1]:5.2f}%  '
          f'Mixte={accuracies[k][2]:5.2f}%  | {train_time_min[k]:5.2f} min  | {cost_total_ca[k]:>8} CA')

## Génération de la figure PNG pour le rapport LaTeX

Cette cellule relit les 3 JSON baselines et sauvegarde un PNG haute résolution
dans `latek_report/figures/baselines_comparison_food101.png`.

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt

baseline_keys   = ['HF', 'BF', 'MIXTE']
baseline_labels = {'HF': 'BL 1(HF)', 'BF': 'BL 2(BF)', 'MIXTE': 'BL 3(MIXTE)'}

candidate_src_dirs = [
    '.',
    '../notebooks',
    '/content/drive/MyDrive/UTBM_PF22/results/Food101',
    '../results/Food101',
]
src_dir = None
for d in candidate_src_dirs:
    if all(os.path.exists(os.path.join(d, f'results_baseline_{k}.json'))
           for k in baseline_keys):
        src_dir = d
        break
if src_dir is None:
    raise FileNotFoundError('Aucun dossier ne contient les 3 JSON. Chemins testes : '
                            + ' | '.join(candidate_src_dirs))
print(f'Lecture des JSON depuis : {os.path.abspath(src_dir)}')

data = {}
for k in baseline_keys:
    with open(os.path.join(src_dir, f'results_baseline_{k}.json'), 'r', encoding='utf-8') as f:
        data[k] = json.load(f)

losses         = {k: data[k].get('loss_history', []) for k in baseline_keys}
accuracies     = {k: [data[k]['accuracy_HF'], data[k]['accuracy_BF'], data[k]['accuracy_Mixte']]
                  for k in baseline_keys}
train_time_min = {k: data[k]['training_time_sec'] / 60.0 for k in baseline_keys}
cost_total_ca  = {k: data[k]['total_cost_CA'] for k in baseline_keys}
acc_labels     = ['Test HF', 'Test BF', 'Test Mixte']
has_losses     = any(len(losses[k]) > 0 for k in baseline_keys)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Comparaison des 3 baselines — Food-101 (10 classes)', fontsize=16)

ax = axes[0, 0]
if has_losses:
    for k in baseline_keys:
        y = losses[k]
        if len(y) > 0:
            ax.plot(np.arange(1, len(y) + 1), y, marker='o', linewidth=2, label=baseline_labels[k])
    ax.legend()
else:
    ax.text(0.5, 0.5, 'loss_history absent du JSON', ha='center', va='center',
            transform=ax.transAxes, fontsize=11, color='gray')
ax.set_title('Loss par époque')
ax.set_xlabel('Époque')
ax.set_ylabel('Loss')
ax.grid(alpha=0.3)

ax = axes[0, 1]
x = np.arange(len(acc_labels))
width = 0.25
for i, k in enumerate(baseline_keys):
    vals = accuracies[k]
    bars = ax.bar(x + (i - 1) * width, vals, width=width, label=baseline_labels[k])
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.6, f'{h:.1f}%',
                ha='center', va='bottom', fontsize=8)
ax.set_title('Précisions sur les jeux de test')
ax.set_xticks(x)
ax.set_xticklabels(acc_labels)
ax.set_ylabel('Précision (%)')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
ax.legend()

ax = axes[1, 0]
time_vals = [train_time_min[k] for k in baseline_keys]
bars = ax.bar([baseline_labels[k] for k in baseline_keys], time_vals,
              color=['#4C72B0', '#55A868', '#C44E52'])
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h, f'{h:.2f} min',
            ha='center', va='bottom', fontsize=9)
ax.set_title("Temps d'entraînement")
ax.set_ylabel('Minutes')
ax.set_ylim(0, max(time_vals) * 1.15 if max(time_vals) > 0 else 1)
ax.grid(axis='y', alpha=0.3)

ax = axes[1, 1]
cost_vals = [cost_total_ca[k] for k in baseline_keys]
bars = ax.bar([baseline_labels[k] for k in baseline_keys], cost_vals,
              color=['#8172B2', '#CCB974', '#64B5CD'])
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h,
            f'{int(h):,} CA'.replace(',', ' '), ha='center', va='bottom', fontsize=9)
ax.set_title('Coût total estimé')
ax.set_ylabel('CA')
ax.set_ylim(0, max(cost_vals) * 1.15 if max(cost_vals) > 0 else 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])

candidate_out_dirs = ['../latek_report/figures', 'latek_report/figures', './figures']
out_dir = next((d for d in candidate_out_dirs if os.path.isdir(d)), src_dir)
out_path = os.path.join(out_dir, 'baselines_comparison_food101.png')
plt.savefig(out_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Figure sauvee : {os.path.abspath(out_path)}')